# Exercise 2: Save Deduplicated Results to Apache Iceberg

## Learning Objectives

In this exercise, you will:
- Load either Exercise 1 Parquet **or** the project local CSV
- Optionally write a **local** Iceberg table (lab / session only)
- Publish a table into the **shared CDW Hive warehouse** so it appears in **Hue**
- Verify via Spark SQL and/or the Hive JDBC connection

## Prerequisites

1. Run **`00_Getting_Started.ipynb`**. For the Parquet path, also run **`01_Basic_Deduplication.ipynb`**.
2. Input options (set `INPUT_SOURCE` in Step 1):

| `INPUT_SOURCE` | File | Table name (default) |
|----------------|------|----------------------|
| `exercise1_parquet` | `/tmp/.../exercise1_exact.parquet` | `deduped_customers` |
| `local_csv` | `../data/redundant_data.csv` | `raw_customers` |

3. Run **Step 0** first: enter JDBC URL + workload password or Knox/CDP JWT for the Hive VW Data Connection.
4. Local `/tmp` Iceberg warehouses are **not** visible in Hue — only tables registered in the shared Hive Metastore / CDW.


## Step 0: Hive Connection Form (JDBC + Auth)

Fill in auth for the shared CDW / Hue Hive connection **before** configuring inputs.

| Field | Purpose |
|-------|--------|
| **Connection name** | CAI Data Connection (auth wiring) |
| **JDBC URL** | CDW HS2 URL — used for **direct Impyla** when CML gateway DNS is broken |
| **Gateway** | Optional alternate host |
| **Username / Password / JWT** | Hive auth (JWT can be used as password) |

**Known issue:** `your-cai-hive-connection` is baked to a non-resolvable gateway host (does not resolve) and **ignores HOSTNAME overrides**.
When that happens, Step 0 opens Hive with **direct Impyla** to your CDW HS2 host from the JDBC URL.

Use **Save connection form**, then **Test Hive connection**.


In [ ]:
import os
import getpass
import socket
from pathlib import Path as _Path
from urllib.parse import unquote

DEFAULT_CONNECTION_NAME = os.environ.get("CML_CONNECTION_NAME", "")
DEFAULT_HIVE_JDBC_URL = os.environ.get(
    "HIVE_JDBC_URL",
    "",  # paste CDW VW "Copy JDBC URL" in the form below
)
JWT_FILE = _Path(os.environ.get("CML_JWT_FILE", "/tmp/cdp_user_demo_knox.jwt"))

CONNECTION_NAME = DEFAULT_CONNECTION_NAME
# When False (recommended), get_connection uses the Project Data Connection endpoint
# instead of the JDBC host — CAI often cannot DNS-resolve CDW hs2-*.dw-* hostnames.
OVERRIDE_ENDPOINT_FROM_JDBC = False
# Optional Knox/CDP gateway host (from JWT jku) — only used if it resolves and override is ON
DEFAULT_GATEWAY_HOST = os.environ.get("CDP_GATEWAY_HOST", "")
GATEWAY_HOST = DEFAULT_GATEWAY_HOST

WORKLOAD_CREDS = {
    "CONNECTION_NAME": CONNECTION_NAME,
    "USERNAME": os.environ.get("WORKLOAD_USER")
    or os.environ.get("KRB_USER")
    or os.environ.get("HADOOP_USER_NAME")
    or "",
    "PASSWORD": os.environ.get("WORKLOAD_PASSWORD") or "",
    "JWT": (
        os.environ.get("CML_JWT")
        or os.environ.get("CDP_TOKEN")
        or os.environ.get("KNOX_TOKEN")
        or ""
    ),
    "JDBC_URL": os.environ.get("HIVE_JDBC_URL") or DEFAULT_HIVE_JDBC_URL,
    "OVERRIDE_ENDPOINT": OVERRIDE_ENDPOINT_FROM_JDBC,
    "GATEWAY_HOST": GATEWAY_HOST,
}
if not WORKLOAD_CREDS["JWT"] and JWT_FILE.is_file():
    WORKLOAD_CREDS["JWT"] = JWT_FILE.read_text().strip()


def parse_hive_jdbc_url(url: str) -> dict:
    """Extract HOSTNAME / HTTP_PATH / DATABASE from a jdbc:hive2:// URL."""
    parsed = {"JDBC_URL": (url or "").strip()}
    u = parsed["JDBC_URL"]
    if not u.lower().startswith("jdbc:hive2://"):
        return parsed
    rest = u[len("jdbc:hive2://") :]
    host_db, _, param_str = rest.partition(";")
    host_port, _, database = host_db.partition("/")
    if ":" in host_port and not host_port.startswith("["):
        host, port = host_port.rsplit(":", 1)
        if port.isdigit():
            parsed["HOSTNAME"] = host
            parsed["PORT"] = port
        else:
            parsed["HOSTNAME"] = host_port
    else:
        parsed["HOSTNAME"] = host_port
    if database:
        parsed["DATABASE"] = database.split(";")[0] or "default"
    for piece in param_str.split(";"):
        piece = piece.strip()
        if not piece or "=" not in piece:
            continue
        k, v = piece.split("=", 1)
        k, v = k.strip(), unquote(v.strip())
        parsed[k] = v
        if k.lower() == "httppath":
            parsed["HTTP_PATH"] = v
        if k.lower() == "transportmode":
            parsed["TRANSPORT_MODE"] = v
    return parsed


def resolve_host(host: str, port: int = 443):
    """Return (ok, detail) for DNS resolution."""
    if not host:
        return False, "empty host"
    try:
        infos = socket.getaddrinfo(host, port, type=socket.SOCK_STREAM)
        addrs = sorted({i[4][0] for i in infos})
        return True, ", ".join(addrs[:3])
    except socket.gaierror as e:
        return False, str(e)


def diagnose_jdbc_host(jdbc_url: str) -> bool:
    """Return True if JDBC host resolves via DNS from this session."""
    host = parse_hive_jdbc_url(jdbc_url).get("HOSTNAME")
    ok, detail = resolve_host(host)
    if ok:
        print(f"✓ DNS OK for JDBC host {host} → {detail}")
    else:
        print(
            f"⚠ DNS FAILED for JDBC host {host}: {detail}\n"
            "  Direct CDW hs2-*.dw-* names often do not resolve from CAI.\n"
            "  Keep endpoint override OFF (use CML Data Connection), or try Gateway host if it resolves."
        )
    return ok


def diagnose_connection_endpoint(hive_conn) -> bool:
    """DNS-check the hostname the CML HiveConnection will actually dial."""
    host = (
        getattr(hive_conn, "hostname", None)
        or getattr(hive_conn, "host", None)
        or (getattr(hive_conn, "properties", {}) or {}).get("HOSTNAME")
    )
    port = getattr(hive_conn, "port", None) or 443
    try:
        port = int(port)
    except Exception:
        port = 443
    print(f"CML connection endpoint host={host!r} port={port}")
    ok, detail = resolve_host(host, port)
    if ok:
        print(f"✓ DNS OK for connection host {host} → {detail}")
    else:
        print(
            f"✗ DNS FAILED for connection host {host}: {detail}\n"
            "  The Project Data Connection hostname is not resolvable from this CAI session.\n"
            "  Fix outside the notebook:\n"
            "    • Start/resume the CDW Virtual Warehouse\n"
            "    • In Project Settings → Data Connections, re-test `<CML-HIVE-CONNECTION>`\n"
            "    • Confirm CAI and CDW are in the same CDP env / network path\n"
            "    • Try Gateway host override only if that hostname resolves here"
        )
    return ok


def test_cml_hive_connection():
    """Smoke-test: connect + SHOW DATABASES. Returns True on success."""
    try:
        c = get_cml_connection()
        if not diagnose_connection_endpoint(c):
            try:
                c.close()
            except Exception:
                pass
            return False
        df = c.get_pandas_dataframe("SHOW DATABASES")
        print("✓ Hive connectivity OK — SHOW DATABASES:")
        print(df)
        try:
            c.close()
        except Exception:
            pass
        return True
    except Exception as e:
        print(f"✗ Hive connectivity FAILED: {type(e).__name__}: {e}")
        return False


def _apply_saved_settings(
    connection_name: str,
    username: str,
    password: str,
    jdbc_url: str,
    jwt: str = "",
    override_endpoint: bool = False,
    gateway_host: str = "",
):
    global CONNECTION_NAME, HIVE_JDBC_URL, OVERRIDE_ENDPOINT_FROM_JDBC, GATEWAY_HOST
    CONNECTION_NAME = (connection_name or "").strip() or DEFAULT_CONNECTION_NAME
    OVERRIDE_ENDPOINT_FROM_JDBC = bool(override_endpoint)
    GATEWAY_HOST = (gateway_host or "").strip() or DEFAULT_GATEWAY_HOST
    WORKLOAD_CREDS["CONNECTION_NAME"] = CONNECTION_NAME
    WORKLOAD_CREDS["USERNAME"] = (username or "").strip()
    WORKLOAD_CREDS["PASSWORD"] = password or ""
    WORKLOAD_CREDS["JWT"] = (jwt or "").strip()
    WORKLOAD_CREDS["JDBC_URL"] = (jdbc_url or "").strip() or DEFAULT_HIVE_JDBC_URL
    WORKLOAD_CREDS["OVERRIDE_ENDPOINT"] = OVERRIDE_ENDPOINT_FROM_JDBC
    WORKLOAD_CREDS["GATEWAY_HOST"] = GATEWAY_HOST
    HIVE_JDBC_URL = WORKLOAD_CREDS["JDBC_URL"]
    os.environ["CML_CONNECTION_NAME"] = CONNECTION_NAME
    os.environ["HIVE_JDBC_URL"] = HIVE_JDBC_URL
    os.environ["CDP_GATEWAY_HOST"] = GATEWAY_HOST
    if WORKLOAD_CREDS["USERNAME"]:
        os.environ["WORKLOAD_USER"] = WORKLOAD_CREDS["USERNAME"]
    if WORKLOAD_CREDS["PASSWORD"]:
        os.environ["WORKLOAD_PASSWORD"] = WORKLOAD_CREDS["PASSWORD"]
    if WORKLOAD_CREDS["JWT"]:
        os.environ["CML_JWT"] = WORKLOAD_CREDS["JWT"]
        os.environ["CDP_TOKEN"] = WORKLOAD_CREDS["JWT"]
        os.environ["KNOX_TOKEN"] = WORKLOAD_CREDS["JWT"]
        JWT_FILE.parent.mkdir(parents=True, exist_ok=True)
        JWT_FILE.write_text(WORKLOAD_CREDS["JWT"])
        os.environ["CML_JWT_FILE"] = str(JWT_FILE)
        os.environ.setdefault("JWT_TOKEN_PATH", str(JWT_FILE))
    parsed = parse_hive_jdbc_url(HIVE_JDBC_URL)
    print("--- DNS checks ---")
    diagnose_jdbc_host(HIVE_JDBC_URL)
    g_ok, g_detail = resolve_host(GATEWAY_HOST)
    if g_ok:
        print(f"✓ DNS OK for gateway {GATEWAY_HOST} → {g_detail}")
    else:
        print(f"⚠ DNS FAILED for gateway {GATEWAY_HOST}: {g_detail}")
    return parsed


try:
    import ipywidgets as widgets
    from IPython.display import display

    _conn_w = widgets.Text(
        value=WORKLOAD_CREDS["CONNECTION_NAME"],
        description="Connection:",
        placeholder="your-cai-hive-connection",
        style={"description_width": "140px"},
        layout=widgets.Layout(width="480px"),
    )
    _jdbc_w = widgets.Textarea(
        value=WORKLOAD_CREDS["JDBC_URL"],
        description="JDBC URL:",
        placeholder="jdbc:hive2://<hs2-host>/default;transportMode=http;httpPath=cliservice;ssl=true;",
        style={"description_width": "140px"},
        layout=widgets.Layout(width="95%", height="72px"),
    )
    _gw_w = widgets.Text(
        value=WORKLOAD_CREDS.get("GATEWAY_HOST", DEFAULT_GATEWAY_HOST),
        description="Gateway:",
        placeholder="knox/cdp gateway hostname",
        style={"description_width": "140px"},
        layout=widgets.Layout(width="95%"),
    )
    _user_w = widgets.Text(
        value=WORKLOAD_CREDS["USERNAME"],
        description="Username:",
        placeholder="CDP workload user",
        style={"description_width": "140px"},
        layout=widgets.Layout(width="480px"),
    )
    _pass_w = widgets.Password(
        value=WORKLOAD_CREDS["PASSWORD"],
        description="Password:",
        placeholder="optional if JWT set",
        style={"description_width": "140px"},
        layout=widgets.Layout(width="480px"),
    )
    _jwt_w = widgets.Textarea(
        value=WORKLOAD_CREDS["JWT"],
        description="JWT:",
        placeholder="Paste Knox/CDP JWT (eyJ...). Leave blank to use workload password.",
        style={"description_width": "140px"},
        layout=widgets.Layout(width="95%", height="110px"),
    )
    _override_w = widgets.Checkbox(
        value=False,
        description="Force gateway HOSTNAME (normally leave OFF; JDBC HS2 auto-fallback is used)",
        indent=False,
    )
    _status = widgets.HTML(
        value="<i>Save auth, then optionally click Test Hive connection.</i>"
    )
    _btn = widgets.Button(description="Save connection form", button_style="primary")
    _test_btn = widgets.Button(description="Test Hive connection", button_style="info")

    def _save_creds(_=None):
        parsed = _apply_saved_settings(
            _conn_w.value,
            _user_w.value,
            _pass_w.value,
            _jdbc_w.value,
            _jwt_w.value,
            _override_w.value,
            _gw_w.value,
        )
        jwt_set = bool(WORKLOAD_CREDS["JWT"])
        pw_set = bool(WORKLOAD_CREDS["PASSWORD"])
        _status.value = (
            f"<b>✓ Saved</b> connection=<code>{CONNECTION_NAME}</code>; "
            f"override_gateway={OVERRIDE_ENDPOINT_FROM_JDBC}; "
            f"user=<code>{WORKLOAD_CREDS['USERNAME'] or '(empty)'}</code>; "
            f"password={'set' if pw_set else 'no'}; JWT={'set' if jwt_set else 'no'}"
        )
        print(
            f"✓ Saved connection={CONNECTION_NAME!r} "
            f"override_gateway={OVERRIDE_ENDPOINT_FROM_JDBC} "
            f"gateway={GATEWAY_HOST!r} "
            f"user={WORKLOAD_CREDS['USERNAME']!r}"
        )

    def _test(_=None):
        _save_creds()
        test_cml_hive_connection()

    _btn.on_click(_save_creds)
    _test_btn.on_click(_test)
    display(
        widgets.VBox(
            [
                widgets.HTML("<b>Shared Hive / Hue connection</b>"),
                _conn_w,
                _jdbc_w,
                _gw_w,
                _user_w,
                _pass_w,
                _jwt_w,
                _override_w,
                widgets.HBox([_btn, _test_btn]),
                _status,
            ]
        )
    )
    _save_creds()
except ImportError:
    print("ipywidgets not available — using prompts instead")
    _c = input(f"Connection name [{WORKLOAD_CREDS['CONNECTION_NAME']}]: ").strip()
    _j = input(f"JDBC URL [{WORKLOAD_CREDS['JDBC_URL'][:60]}...]: ").strip()
    _g = input(f"Gateway host [{DEFAULT_GATEWAY_HOST}]: ").strip()
    _u = input(f"Workload username [{WORKLOAD_CREDS['USERNAME']}]: ").strip()
    if _c:
        WORKLOAD_CREDS["CONNECTION_NAME"] = _c
    if _j:
        WORKLOAD_CREDS["JDBC_URL"] = _j
    if _u:
        WORKLOAD_CREDS["USERNAME"] = _u
    use_jwt = input("Auth with JWT instead of password? [Y/n]: ").strip().lower() not in (
        "n",
        "no",
    )
    if use_jwt:
        WORKLOAD_CREDS["JWT"] = getpass.getpass("Paste JWT (input hidden): ").strip()
    elif not WORKLOAD_CREDS["PASSWORD"]:
        WORKLOAD_CREDS["PASSWORD"] = getpass.getpass("Workload password: ")
    _apply_saved_settings(
        WORKLOAD_CREDS["CONNECTION_NAME"],
        WORKLOAD_CREDS["USERNAME"],
        WORKLOAD_CREDS["PASSWORD"],
        WORKLOAD_CREDS["JDBC_URL"],
        WORKLOAD_CREDS["JWT"],
        False,
        _g or DEFAULT_GATEWAY_HOST,
    )


class DirectHiveConnection:
    """Impyla HiveServer2 client using the JDBC host (bypasses broken CML gateway DNS)."""

    def __init__(self, host, user, password, http_path="cliservice", port=443, database="default"):
        from impala.dbapi import connect

        self.hostname = host
        self.port = port
        self.http_path = http_path
        self.database = database or "default"
        # CDW Hive over HTTPS + LDAP/password (JWT often accepted as password)
        self._conn = connect(
            host=host,
            port=int(port),
            user=user or None,
            password=password or None,
            database=self.database,
            auth_mechanism="LDAP",
            use_ssl=True,
            use_http_transport=True,
            http_path=http_path,
        )

    def get_cursor(self):
        return self._conn.cursor()

    def get_pandas_dataframe(self, query):
        import pandas as pd

        cur = self.get_cursor()
        try:
            cur.execute(query)
            cols = [d[0] for d in (cur.description or [])]
            rows = cur.fetchall()
            return pd.DataFrame.from_records(rows, columns=cols or None)
        finally:
            try:
                cur.close()
            except Exception:
                pass

    def close(self):
        try:
            self._conn.close()
        except Exception:
            pass


def _auth_material():
    user = (WORKLOAD_CREDS.get("USERNAME") or os.environ.get("WORKLOAD_USER") or "").strip()
    password = WORKLOAD_CREDS.get("PASSWORD") or os.environ.get("WORKLOAD_PASSWORD") or ""
    jwt = (
        WORKLOAD_CREDS.get("JWT")
        or os.environ.get("CML_JWT")
        or os.environ.get("CDP_TOKEN")
        or os.environ.get("KNOX_TOKEN")
        or ""
    ).strip()
    secret = password or jwt
    return user, password, jwt, secret


def get_cml_connection(connection_name: str = None):
    """Open Hive: try CML Data Connection, then direct Impyla to JDBC HS2 host.

    Some CML Hive connections ignore HOSTNAME overrides and keep an unresolvable
    gateway host. Direct Impyla to the JDBC HS2 host is used when DNS allows.
    """
    import cml.data_v1 as cmldata

    name = connection_name or CONNECTION_NAME or DEFAULT_CONNECTION_NAME
    if not name:
        raise ValueError(
            "Set CML connection name in Step 0 (Project Settings → Data Connections)."
        )
    jdbc_url = (
        WORKLOAD_CREDS.get("JDBC_URL")
        or os.environ.get("HIVE_JDBC_URL")
        or globals().get("HIVE_JDBC_URL")
        or DEFAULT_HIVE_JDBC_URL
    )
    parsed = parse_hive_jdbc_url(jdbc_url)
    user, password, jwt, secret = _auth_material()
    jdbc_host = parsed.get("HOSTNAME")
    if not jdbc_host or jdbc_host.startswith("<"):
        # still allow CML-only path; direct Impyla needs a real JDBC host
        pass
    http_path = parsed.get("HTTP_PATH") or "cliservice"
    database = parsed.get("DATABASE") or "default"
    gateway = (
        WORKLOAD_CREDS.get("GATEWAY_HOST")
        or globals().get("GATEWAY_HOST")
        or os.environ.get("CDP_GATEWAY_HOST")
        or DEFAULT_GATEWAY_HOST
    )

    if not secret:
        raise KeyError(
            "No workload password or JWT set. Fill Step 0 (Save connection form)."
        )

    # Persist JWT file for any CML JWT-file based paths
    if jwt:
        JWT_FILE.parent.mkdir(parents=True, exist_ok=True)
        JWT_FILE.write_text(jwt)
        os.environ["CML_JWT"] = jwt
        os.environ["CDP_TOKEN"] = jwt
        os.environ["CML_JWT_FILE"] = str(JWT_FILE)
        os.environ["JWT_TOKEN_PATH"] = str(JWT_FILE)
    if password:
        os.environ["WORKLOAD_PASSWORD"] = password

    def _cml_params():
        params = {}
        if user:
            params["USERNAME"] = user
        params["PASSWORD"] = secret
        if jwt:
            params["TOKEN"] = jwt
            params["JWT"] = jwt
            params["ACCESS_TOKEN"] = jwt
        return params

    def _cml_host(conn):
        return (
            getattr(conn, "hostname", None)
            or getattr(conn, "host", None)
            or (getattr(conn, "properties", {}) or {}).get("HOSTNAME")
        )

    # 1) CML Data Connection (may have broken gateway hostname baked in)
    try:
        print(
            f"Connecting {name!r} → CML Data Connection defaults "
            f"user={user!r} auth={'JWT' if jwt and not password else 'PASSWORD'}"
        )
        conn = cmldata.get_connection(name, _cml_params())
        host = _cml_host(conn)
        ok, detail = resolve_host(host)
        print(f"CML connection endpoint host={host!r} → {'OK ' + detail if ok else 'DNS FAIL: ' + detail}")
        if ok:
            # Try mutating hostname if CML kept a bad one despite params — already OK
            print("✓ Using CML Data Connection endpoint")
            return conn
        # Attempt in-place hostname repair (often ignored by already-built thrift client)
        for alt in (jdbc_host, gateway):
            if not alt:
                continue
            alt_ok, _ = resolve_host(alt)
            if not alt_ok:
                continue
            try:
                if hasattr(conn, "hostname"):
                    conn.hostname = alt
                props = getattr(conn, "properties", None)
                if isinstance(props, dict):
                    props["HOSTNAME"] = alt
                    if http_path:
                        props["HTTP_PATH"] = http_path
                if hasattr(conn, "update_properties"):
                    conn.update_properties({"HOSTNAME": alt, "HTTP_PATH": http_path, **_cml_params()})
                # Force rebuild of underlying DBAPI connection if present
                if hasattr(conn, "conn"):
                    try:
                        conn.conn.close()
                    except Exception:
                        pass
                    conn.conn = None
                print(f"⚠ CML host {host!r} unresolvable; attempted in-place switch to {alt!r}")
            except Exception as e:
                print(f"⚠ Could not repair CML connection object: {e}")
            break
        try:
            conn.close()
        except Exception:
            pass
    except Exception as e:
        print(f"⚠ CML get_connection failed: {type(e).__name__}: {e}")

    # 2) Direct Impyla to JDBC HS2 (this is the path that matches your working DNS)
    if jdbc_host:
        j_ok, j_detail = resolve_host(jdbc_host)
        if j_ok:
            print(
                f"Connecting direct Impyla → host={jdbc_host!r} httpPath={http_path!r} "
                f"user={user!r} ({j_detail})"
            )
            direct = DirectHiveConnection(
                host=jdbc_host,
                user=user,
                password=secret,
                http_path=http_path,
                port=443,
                database=database,
            )
            print("✓ Using direct JDBC HS2 Impyla connection (CML gateway hostname bypassed)")
            return direct
        print(f"⚠ JDBC host {jdbc_host!r} DNS failed: {j_detail}")

    # 3) Last resort: Impyla via gateway host
    g_ok, g_detail = resolve_host(gateway)
    if g_ok:
        print(
            f"Connecting direct Impyla → gateway={gateway!r} httpPath={http_path!r} "
            f"user={user!r} ({g_detail})"
        )
        direct = DirectHiveConnection(
            host=gateway,
            user=user,
            password=secret,
            http_path=http_path,
            port=443,
            database=database,
        )
        print("✓ Using direct gateway Impyla connection")
        return direct

    raise RuntimeError(
        "No usable Hive endpoint. CML connection host does not resolve, and JDBC/gateway "
        "direct connect is unavailable. Fix Data Connection hostname or network DNS."
    )


## Step 1: Configure Connection and Input Source

Uses the CAI connection / JDBC / auth from **Step 0**. Choose `INPUT_SOURCE` to load Exercise 1 Parquet or the project local CSV (writes a differently named Iceberg table).


In [ ]:
import os
from pathlib import Path

# --- CAI connection name from Step 0 form (fallback default) ---
CONNECTION_NAME = (
    globals().get("CONNECTION_NAME")
    or os.environ.get("CML_CONNECTION_NAME")
    or ""
)

# --- Input source: "exercise1_parquet" (default) or "local_csv" ---
INPUT_SOURCE = os.environ.get("ICEBERG_INPUT_SOURCE", "exercise1_parquet").strip().lower()
# Flip to local CSV without env vars:
# INPUT_SOURCE = "local_csv"

NAMESPACE = os.environ.get("ICEBERG_NAMESPACE", "cdp_user_demo")

# Distinct Iceberg table names per source
TABLE_NAME_DEDUPED = os.environ.get("ICEBERG_TABLE_DEDUPED", "deduped_customers")
TABLE_NAME_RAW = os.environ.get("ICEBERG_TABLE_RAW", "raw_customers")

def _resolve_local_csv() -> Path:
    env = os.environ.get("LOCAL_CSV")
    if env:
        return Path(env).resolve()
    candidates = [
        Path("../data/redundant_data.csv"),
        Path("data/redundant_data.csv"),
        Path.cwd() / "data" / "redundant_data.csv",
        Path.cwd().parent / "data" / "redundant_data.csv",
        Path.cwd() / "use-case-phase-1" / "data" / "redundant_data.csv",
    ]
    for p in candidates:
        if p.is_file():
            return p.resolve()
    return Path("../data/redundant_data.csv").resolve()


LOCAL_CSV = _resolve_local_csv()
LOCAL_PARQUET = Path(
    os.environ.get(
        "EXERCISE1_PARQUET",
        "/tmp/cdp_user_demo/phase1/results/exercise1_exact.parquet",
    )
)

# Auto-fallback: Exercise 1 output lives under /tmp and disappears after session restart
if INPUT_SOURCE in ("exercise1_parquet", "parquet") and not LOCAL_PARQUET.exists():
    if LOCAL_CSV.exists():
        print(
            f"⚠ {LOCAL_PARQUET} not found (re-run notebook 01, or it was cleared with /tmp).\n"
            f"  Falling back to local CSV → Iceberg table '{TABLE_NAME_RAW}'."
        )
        INPUT_SOURCE = "local_csv"
    else:
        raise FileNotFoundError(
            f"Missing Exercise 1 Parquet ({LOCAL_PARQUET}) and local CSV ({LOCAL_CSV}).\n"
            "Re-run 01_Basic_Deduplication.ipynb or ensure data/redundant_data.csv is in the project."
        )

if INPUT_SOURCE == "local_csv":
    INPUT_FORMAT = "csv"
    LOCAL_INPUT = LOCAL_CSV
    TABLE_NAME = TABLE_NAME_RAW
elif INPUT_SOURCE in ("exercise1_parquet", "parquet"):
    INPUT_FORMAT = "parquet"
    LOCAL_INPUT = LOCAL_PARQUET
    TABLE_NAME = TABLE_NAME_DEDUPED
else:
    raise ValueError(
        f"Unknown INPUT_SOURCE={INPUT_SOURCE!r}. Use 'exercise1_parquet' or 'local_csv'."
    )

# Optional override of the resolved table name
TABLE_NAME = os.environ.get("ICEBERG_TABLE", TABLE_NAME)
FULL_TABLE = f"{NAMESPACE}.{TABLE_NAME}"
INPUT_PATH = LOCAL_INPUT.resolve().as_uri() if LOCAL_INPUT.exists() else str(LOCAL_INPUT)

# --- Shared CDW / Hue (HiveServer2 JDBC via CML Data Connection) ---
# Connection snippet JDBC (reference — CML connection already embeds this):
# Paste your CDW Hive JDBC URL from the Virtual Warehouse UI (Copy JDBC URL).
HIVE_JDBC_URL = (
    globals().get("HIVE_JDBC_URL")
    or os.environ.get("HIVE_JDBC_URL")
    or ""
)
# Publish into shared warehouse so Hue can see it (set False to skip Step 4b)
WRITE_TO_SHARED = os.environ.get("WRITE_TO_SHARED", "true").lower() in ("1", "true", "yes")
# Distinct Hue-facing name so it is obvious vs local-only lab tables
SHARED_TABLE_NAME = os.environ.get(
    "ICEBERG_SHARED_TABLE",
    f"{TABLE_NAME}_shared",
)
SHARED_FULL_TABLE = f"{NAMESPACE}.{SHARED_TABLE_NAME}"

# --- Optional: manual Iceberg REST catalog (cluster) ---
CATALOG_NAME = os.environ.get("ICEBERG_CATALOG_NAME", "iceberg")
REST_URI = os.environ.get("ICEBERG_REST_URI", "").strip()
REST_CREDENTIAL = os.environ.get("ICEBERG_REST_CREDENTIAL", "").strip()
REST_URI_CONFIGURED = bool(REST_URI) and "<" not in REST_URI and ">" not in REST_URI

# --- Local Iceberg Hadoop warehouse (session-only; NOT visible in Hue) ---
LOCAL_CATALOG = os.environ.get("ICEBERG_LOCAL_CATALOG", "local")
LOCAL_WAREHOUSE = Path(
    os.environ.get(
        "ICEBERG_LOCAL_WAREHOUSE",
        "/tmp/cdp_user_demo/phase1/iceberg-warehouse",
    )
).resolve()

print(f"CML connection: {CONNECTION_NAME}")
print(f"Input source:   {INPUT_SOURCE} ({INPUT_FORMAT})")
print(f"Input path:     {INPUT_PATH}")
print(f"Input exists:   {LOCAL_INPUT.exists()}")
print(f"Local table:    {FULL_TABLE} (session warehouse only)")
print(f"Shared/Hue:     {SHARED_FULL_TABLE} (WRITE_TO_SHARED={WRITE_TO_SHARED})")
print(f"Hive JDBC host: {HIVE_JDBC_URL.split('://')[1].split('/')[0] if '://' in HIVE_JDBC_URL else HIVE_JDBC_URL}")
print(f"Local warehouse:{LOCAL_WAREHOUSE}")
print(f"REST URI set:   {REST_URI_CONFIGURED}")


## Step 2: Create Spark Session

1. Try `cml.data_v1.get_connection(...).get_spark_session()` (Spark Data Lake connection).
2. Else local Spark + **local Hadoop Iceberg warehouse** under `/tmp/.../iceberg-warehouse` (works even if `your-cai-hive-connection` is SQL-only / pandas).
3. Optional: attach Iceberg REST catalog when `ICEBERG_REST_URI` is a real URL — never with a `<placeholder>`.


In [ ]:
import os
import pyspark
from pyspark.sql import SparkSession

spark = None
conn = None
SPARK_MODE = None  # "cml_spark" | "local_rest" | "local_hadoop"

# 1) Preferred: CAI Data Connection Spark session (Iceberg + cluster Hadoop already wired)
try:
    conn = get_cml_connection(CONNECTION_NAME)
    get_spark = getattr(conn, "get_spark_session", None)
    if callable(get_spark):
        spark = get_spark()
        SPARK_MODE = "cml_spark"
        print(f"✓ Spark from CML connection: {CONNECTION_NAME}")
    else:
        # SQL-only connection (pandas/JDBC) — explore DBs, then use local Iceberg warehouse
        try:
            sample = conn.get_pandas_dataframe("show databases")
            print(f"✓ CML connection '{CONNECTION_NAME}' is SQL-only (pandas). Sample databases:")
            print(sample)
        except Exception as e:
            print(f"⚠ CML connection '{CONNECTION_NAME}' has no get_spark_session: {e}")
        print("→ Falling back to local Spark + local Iceberg Hadoop warehouse")
except ImportError:
    print("⚠ cml.data_v1 not available — falling back to local Spark")
except Exception as e:
    print(f"⚠ CML get_connection failed ({e}) — falling back to local Spark")

# 2) Local Spark + Iceberg catalog (REST if configured, else Hadoop warehouse on local disk)
if spark is None:
    for _k in ("HADOOP_CONF_DIR", "HADOOP_HOME", "HADOOP_HDFS_HOME"):
        if _k in os.environ:
            print(f"⚠ Unsetting {_k}={os.environ[_k]} for local Spark startup")
            os.environ.pop(_k)

    _spark_mm = ".".join(pyspark.__version__.split(".")[:2])
    ICEBERG_PACKAGE = os.environ.get(
        "ICEBERG_SPARK_PACKAGE",
        f"org.apache.iceberg:iceberg-spark-runtime-{_spark_mm}_2.12:1.6.1",
    )

    builder = (
        SparkSession.builder
        .master("local[*]")
        .appName("Exercise2_Iceberg")
        .config("spark.sql.shuffle.partitions", "8")
        .config("spark.hadoop.hadoop.security.authentication", "simple")
        .config("spark.hadoop.hadoop.security.authorization", "false")
        .config("spark.jars.packages", ICEBERG_PACKAGE)
        .config(
            "spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
        )
    )

    if REST_URI_CONFIGURED:
        builder = (
            builder
            .config("spark.sql.defaultCatalog", CATALOG_NAME)
            .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog")
            .config(f"spark.sql.catalog.{CATALOG_NAME}.type", "rest")
            .config(f"spark.sql.catalog.{CATALOG_NAME}.uri", REST_URI)
            .config(f"spark.sql.catalog.{CATALOG_NAME}.default-namespace", NAMESPACE)
        )
        if REST_CREDENTIAL:
            builder = builder.config(
                f"spark.sql.catalog.{CATALOG_NAME}.credential", REST_CREDENTIAL
            )
        SPARK_MODE = "local_rest"
        globals()["FULL_TABLE"] = f"{CATALOG_NAME}.{NAMESPACE}.{TABLE_NAME}"
        print(f"✓ Local Spark + Iceberg REST catalog '{CATALOG_NAME}'")
        print(f"  REST URI: {REST_URI}")
    else:
        LOCAL_WAREHOUSE.mkdir(parents=True, exist_ok=True)
        builder = (
            builder
            .config("spark.sql.defaultCatalog", LOCAL_CATALOG)
            .config(f"spark.sql.catalog.{LOCAL_CATALOG}", "org.apache.iceberg.spark.SparkCatalog")
            .config(f"spark.sql.catalog.{LOCAL_CATALOG}.type", "hadoop")
            .config(f"spark.sql.catalog.{LOCAL_CATALOG}.warehouse", str(LOCAL_WAREHOUSE))
            .config(f"spark.sql.catalog.{LOCAL_CATALOG}.default-namespace", NAMESPACE)
        )
        SPARK_MODE = "local_hadoop"
        globals()["FULL_TABLE"] = f"{LOCAL_CATALOG}.{NAMESPACE}.{TABLE_NAME}"
        print(f"✓ Local Spark + Iceberg Hadoop catalog '{LOCAL_CATALOG}'")
        print(f"  Warehouse: {LOCAL_WAREHOUSE}")

    spark = builder.getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Spark master:  {spark.sparkContext.master}")
print(f"Mode:          {SPARK_MODE}")
print(f"Write target:  {FULL_TABLE}")


## Step 3: Load Source Data

Loads based on `INPUT_SOURCE` from Step 1:
- `exercise1_parquet` → Exercise 1 deduped Parquet (if missing under `/tmp`, Step 1 auto-falls back to CSV)
- `local_csv` → project `../data/redundant_data.csv` → table `raw_customers`


In [ ]:
assert LOCAL_INPUT.exists(), (
    f"Missing input for INPUT_SOURCE={INPUT_SOURCE!r}: {LOCAL_INPUT}\n"
    + (
        "Re-run 01_Basic_Deduplication.ipynb, or set EXERCISE1_PARQUET."
        if INPUT_FORMAT == "parquet"
        else "Ensure use-case-phase-1/data/redundant_data.csv is in the project, or set LOCAL_CSV."
    )
)

if INPUT_FORMAT == "csv":
    df = spark.read.csv(INPUT_PATH, header=True, inferSchema=True)
else:
    df = spark.read.parquet(INPUT_PATH)

print(f"✓ Loaded ({INPUT_FORMAT}): {INPUT_PATH}")
print(f"Will write Iceberg table: {FULL_TABLE}")
print(f"Rows: {df.count():,}")
print(f"Columns: {', '.join(df.columns)}")
df.show(10, truncate=False)
df.printSchema()


## Step 4: Create Namespace and Write Iceberg Table

Works with `cml_spark`, `local_rest`, or `local_hadoop` (session-local warehouse under `/tmp/.../iceberg-warehouse`).

Uses `DROP TABLE IF EXISTS` + `create()` so a half-written local table (missing `version-hint.text`) does not break the write. A one-time Hadoop `version-hint` WARN on first create is harmless if the cell still prints success.


In [ ]:
import shutil

assert SPARK_MODE in ("cml_spark", "local_rest", "local_hadoop"), (
    "Unexpected SPARK_MODE for Iceberg write: "
    f"{SPARK_MODE}. Re-run Step 2 after a kernel restart."
)

spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {NAMESPACE}")
print(f"✓ Namespace ready: {NAMESPACE}")

# Prefer drop+create over createOrReplace: Hadoop catalog often WARNs/fails when
# metadata/ exists without version-hint.text (partial prior write under /tmp).
try:
    spark.sql(f"DROP TABLE IF EXISTS {FULL_TABLE}")
except Exception as e:
    print(f"⚠ DROP TABLE noted: {e}")

if SPARK_MODE == "local_hadoop":
    table_dir = LOCAL_WAREHOUSE / NAMESPACE / TABLE_NAME
    if table_dir.exists():
        shutil.rmtree(table_dir)
        print(f"✓ Cleared local table dir: {table_dir}")

(
    df.writeTo(FULL_TABLE)
    .using("iceberg")
    .tableProperty("write.format.default", "parquet")
    .create()
)

n = spark.table(FULL_TABLE).count()
print(f"✓ Iceberg table written: {FULL_TABLE} ({n:,} rows)")
if SPARK_MODE == "local_hadoop":
    meta = LOCAL_WAREHOUSE / NAMESPACE / TABLE_NAME / "metadata"
    hint = meta / "version-hint.text"
    print(f"  Warehouse: {LOCAL_WAREHOUSE}")
    print(f"  version-hint.text: {'present' if hint.is_file() else 'missing'}")


## Step 4b: Publish to Shared CDW Warehouse (visible in Hue)

Publishes via the CML Hive Data Connection so Hue can see the table.

**If you see `gaierror: Name or service not known`:** this CAI session cannot DNS-resolve the Hive VW hostname stored on connection `your-cai-hive-connection` (often `hs2-*.dw-*`). That is an environment/network issue, not a notebook SQL bug.

Checklist:
1. CDW Virtual Warehouse is **running** (not suspended)
2. Project Data Connection points at a host reachable from CAI (re-test the connection snippet in Project Settings)
3. Prefer JWT/password auth in Step 0; keep JDBC host override OFF unless DNS works
4. Optional: set **Gateway host** in Step 0 to the Knox gateway (from your JWT `jku`, from your JWT `jku` host) and enable override only if that host resolves

This step **soft-skips** on DNS/network failure so the local Iceberg lab path can continue.


In [ ]:
import math
import socket

if not WRITE_TO_SHARED:
    print("Skipped — WRITE_TO_SHARED is False")
else:
    hive_conn = None
    try:
        hive_conn = get_cml_connection(CONNECTION_NAME)

        # Preflight: fail soft if CML connection host cannot be resolved
        if not diagnose_connection_endpoint(hive_conn):
            raise socket.gaierror(-2, "Name or service not known (CML Hive endpoint)")

        def hive_type(spark_dtype: str) -> str:
            d = spark_dtype.lower()
            if d.startswith("int"):
                return "BIGINT" if "64" in d or d == "long" else "INT"
            if d.startswith("bigint") or d == "long":
                return "BIGINT"
            if d.startswith("double") or d.startswith("float") or d.startswith("decimal"):
                return "DOUBLE"
            if d.startswith("boolean"):
                return "BOOLEAN"
            if d.startswith("timestamp"):
                return "TIMESTAMP"
            if d.startswith("date"):
                return "DATE"
            return "STRING"

        def sql_literal(value):
            if value is None:
                return "NULL"
            if isinstance(value, bool):
                return "TRUE" if value else "FALSE"
            if isinstance(value, (int, float)) and not isinstance(value, bool):
                if isinstance(value, float) and (math.isnan(value) or math.isinf(value)):
                    return "NULL"
                return str(value)
            s = str(value).replace("\\", "\\\\").replace("'", "''")
            return f"'{s}'"

        col_defs = ", ".join(f"`{c}` {hive_type(t)}" for c, t in df.dtypes)
        create_candidates = [
            f"CREATE TABLE {SHARED_FULL_TABLE} ({col_defs}) USING ICEBERG",
            f"CREATE TABLE {SHARED_FULL_TABLE} ({col_defs}) STORED AS PARQUET",
        ]

        cursor = hive_conn.get_cursor()
        cursor.execute(f"CREATE DATABASE IF NOT EXISTS {NAMESPACE}")
        print(f"✓ Database ready: {NAMESPACE}")

        try:
            cursor.execute(f"DROP TABLE IF EXISTS {SHARED_FULL_TABLE}")
            print(f"✓ Dropped existing {SHARED_FULL_TABLE} (if any)")
        except Exception as e:
            print(f"⚠ DROP TABLE: {e}")

        created = False
        last_err = None
        for create_sql in create_candidates:
            try:
                cursor.execute(create_sql)
                print(f"✓ Created table: {SHARED_FULL_TABLE}")
                print(f"  DDL: {create_sql}")
                created = True
                break
            except Exception as e:
                last_err = e
                print(f"⚠ Create failed, trying next DDL style: {e}")
        if not created:
            raise RuntimeError(f"Could not create shared table: {last_err}")

        cols = df.columns
        col_list = ", ".join(f"`{c}`" for c in cols)
        pdf = df.toPandas()
        batch_size = 100
        inserted = 0
        for start in range(0, len(pdf), batch_size):
            chunk = pdf.iloc[start : start + batch_size]
            values_sql = []
            for row in chunk.itertuples(index=False, name=None):
                values_sql.append("(" + ", ".join(sql_literal(v) for v in row) + ")")
            insert_sql = (
                f"INSERT INTO {SHARED_FULL_TABLE} ({col_list}) VALUES "
                + ", ".join(values_sql)
            )
            cursor.execute(insert_sql)
            inserted += len(chunk)
        print(f"✓ Inserted {inserted:,} rows into {SHARED_FULL_TABLE}")

        tables = hive_conn.get_pandas_dataframe(f"SHOW TABLES IN {NAMESPACE}")
        print("\n=== Tables in shared database (JDBC) ===")
        print(tables)

        preview = hive_conn.get_pandas_dataframe(
            f"SELECT * FROM {SHARED_FULL_TABLE} LIMIT 10"
        )
        print(f"\n=== Preview {SHARED_FULL_TABLE} ===")
        print(preview)
        print(
            f"\n→ In Hue: Table Browser → database `{NAMESPACE}` → "
            f"`{SHARED_TABLE_NAME}` (your Hive Virtual Warehouse)."
        )
    except OSError as e:
        # Includes socket.gaierror
        print(
            f"\n⚠ Step 4b skipped — cannot reach Hive from this CAI session ({type(e).__name__}: {e}).\n"
            "  Local Iceberg tables from Step 4 are unchanged.\n"
            "  Next actions:\n"
            "    1) In Step 0 click **Test Hive connection** and note which host fails DNS\n"
            "    2) Start the CDW Virtual Warehouse if suspended\n"
            "    3) Re-test Data Connection `<CML-HIVE-CONNECTION>` in Project Settings\n"
            "    4) If gateway DNS works, enable 'Override HOSTNAME with Gateway' in Step 0\n"
            "    5) Or create/load the table directly in Hue with the same CSV"
        )
    except Exception as e:
        print(f"\n⚠ Step 4b failed: {type(e).__name__}: {e}")
        print("  Local Iceberg path (Step 4) can still be used for the lab.")
    finally:
        if hive_conn is not None:
            try:
                hive_conn.close()
            except Exception:
                pass


## Step 5: Find the Table in the Catalog

List namespaces/tables and confirm the new table is discoverable.


In [ ]:
print("=== Databases / namespaces ===")
spark.sql("SHOW NAMESPACES").show(truncate=False)

print(f"=== Tables in {NAMESPACE} ===")
tables_df = spark.sql(f"SHOW TABLES IN {NAMESPACE}")
tables_df.show(truncate=False)

rows = tables_df.collect()
table_names = []
for r in rows:
    d = r.asDict()
    table_names.append(d.get("tableName") or d.get("name") or d.get("table") or str(r[0]))

found = TABLE_NAME in table_names or any(TABLE_NAME == str(n) for n in table_names)
print(f"Looking for table: {TABLE_NAME}")
print(f"Tables found: {table_names}")
print("✓ Table found in catalog" if found else "✗ Table NOT found — check write step / namespace")


In [ ]:
print("=== DESCRIBE TABLE ===")
spark.sql(f"DESCRIBE TABLE EXTENDED {FULL_TABLE}").show(100, truncate=False)

print("=== Sample query from Iceberg table ===")
iceberg_df = spark.table(FULL_TABLE)
print(f"Rows in Iceberg table: {iceberg_df.count():,}")
iceberg_df.show(10, truncate=False)


## Step 6 (Optional): Query the REST Catalog HTTP API Directly

Only when `ICEBERG_REST_URI` is configured. Skip for CML Spark mode — use Step 5 Spark SQL instead.


In [ ]:
import json
import urllib.request
import urllib.error
import base64

if not REST_URI_CONFIGURED:
    print("Skipped — ICEBERG_REST_URI not set. Use Step 5 Spark SQL for catalog discovery.")
else:
    def rest_get(path: str):
        """GET a path under the Iceberg REST catalog URI."""
        url = REST_URI.rstrip("/") + path
        req = urllib.request.Request(url, method="GET")
        req.add_header("Accept", "application/json")

        token = os.environ.get("ICEBERG_REST_TOKEN", "")
        if token:
            req.add_header("Authorization", f"Bearer {token}")
        elif REST_CREDENTIAL:
            encoded = base64.b64encode(REST_CREDENTIAL.encode("utf-8")).decode("ascii")
            req.add_header("Authorization", f"Basic {encoded}")

        with urllib.request.urlopen(req, timeout=30) as resp:
            return json.loads(resp.read().decode("utf-8"))

    try:
        ns_path = NAMESPACE.replace(".", "%1F")
        payload = rest_get(f"/v1/namespaces/{ns_path}/tables")
        identifiers = payload.get("identifiers", [])
        print("REST API tables in namespace:")
        print(json.dumps(identifiers, indent=2))

        matched = [
            i for i in identifiers
            if i.get("name") == TABLE_NAME or TABLE_NAME in str(i)
        ]
        if matched:
            print(f"\n✓ Found via REST API: {matched}")
            meta = rest_get(f"/v1/namespaces/{ns_path}/tables/{TABLE_NAME}")
            print("\nTable metadata keys:", list(meta.keys()))
        else:
            print(f"\n✗ Table '{TABLE_NAME}' not returned by REST list endpoint")
    except urllib.error.HTTPError as e:
        print(f"REST API HTTP error: {e.code} {e.reason}")
        print("Spark SQL discovery in Step 5 may still succeed.")
    except Exception as e:
        print(f"REST API call skipped/failed: {e}")


## Summary

| Item | Value |
|------|-------|
| Input A | Exercise 1 Parquet → `deduped_customers` |
| Input B | Local CSV → `raw_customers` |
| Local only | Hadoop Iceberg under `/tmp/.../iceberg-warehouse` (**not** in Hue) |
| Shared / Hue | `cdp_user_demo.<table>_shared` via Hive JDBC / Impyla to your CDW HS2 |
| JDBC | from CDW VW **Copy JDBC URL** (set in Step 0) |

### Key Takeaways

- **Hue only sees shared Metastore / CDW tables**, not CAI session `/tmp` warehouses
- Use the Hive VW Data Connection (`get_cursor` / `get_pandas_dataframe`) to publish lab data for Hue
- Prefer Iceberg (`USING ICEBERG`); notebook falls back to `STORED AS PARQUET` if needed
- Keep distinct names for local vs shared (`*_shared`) so sources do not collide

## Cleanup


In [ ]:
spark.stop()
if conn is not None and hasattr(conn, "close"):
    try:
        conn.close()
    except Exception:
        pass
print("✓ Spark session stopped")
